# Final Three-Stage Showcase

Task 8 Stage 1 followed by the Task 9 general and specialist stages. This notebook uses the Task 7 N sweep and sigma `0.001` training-only Gaussian gradient noise. The Task 9 routing and losses are not replaced by the optimized two-stage recipe.

In [ ]:
import gc,json,sys
from pathlib import Path
ROOT=Path.cwd()
while not (ROOT/'src').exists() and ROOT.parent!=ROOT: ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import pandas as pd
import torch
from config import Task9RunConfig,Task9StackConfig
from final_models.three_stage import run_three_stage
from final_models.sweep import FINAL_SWEEP
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_ROOT=ROOT/'outputs'/'final_three_stage'
FORCE_RERUN=True
torch.set_num_threads(2)
try: torch.set_num_interop_threads(1)
except RuntimeError: pass
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
print('Device:',DEVICE,'Sweep:',FINAL_SWEEP)

In [ ]:
rows=[]
for index,n_value in enumerate(FINAL_SWEEP.n_values):
    config=Task9RunConfig(N=n_value,training_samples=FINAL_SWEEP.training_sizes[index],validation_samples=FINAL_SWEEP.validation_sizes[index],test_samples=FINAL_SWEEP.test_samples,noise_sigma=FINAL_SWEEP.noise_sigma,noise_mode=FINAL_SWEEP.noise_mode,seed=FINAL_SWEEP.seed+index,model=Task9StackConfig(),output_dir=OUTPUT_ROOT)
    path=config.run_output_dir/'final_summary.json'
    if path.exists() and not FORCE_RERUN: result=json.loads(path.read_text())
    else:
        print(f'Running N={n_value}',flush=True); summary=run_three_stage(config,device=DEVICE); result={'N':n_value,'training_samples':config.training_samples,'validation_samples':config.validation_samples,'metrics':summary.get('metrics',{}),'training_summary':summary.get('training_summary',{}),'metrics_by_shape':summary.get('metrics_by_shape',{})}; path.write_text(json.dumps(result,indent=2,default=str),encoding='utf-8'); del summary,result,config; gc.collect();
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        result=json.loads(path.read_text())
    rows.append({'N':n_value,'training_samples':result['training_samples'],'metrics':result['metrics']})
results=pd.DataFrame(rows); results.to_json(OUTPUT_ROOT/'sweep_results.json',orient='records',indent=2); results